In [87]:
from __future__ import annotations

import operator
import os
from pathlib import Path
from typing import TypedDict, List, Annotated

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langgraph.checkpoint.postgres import PostgresSaver

import psycopg
from psycopg.rows import dict_row

load_dotenv()

True

In [88]:
# ============================================================
# 1. LOAD ENVIRONMENT VARIABLES
# ============================================================

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError(
        "GROQ_API_KEY is missing. Please add it to your .env file."
    )

In [ ]:
print(f"GROQ_API_KEY:", GROQ_API_KEY)

In [90]:
# ============================================================
# 2. DATABASE URL
# ============================================================

def get_database_url():
    database_url = os.getenv("DATABASE_URL")

    if not database_url:
        raise ValueError(
            "DATABASE_URL is missing. "
            "Please add your Render PostgreSQL External Database URL to .env"
        )

    if "sslmode=" not in database_url:
        separator = "&" if "?" in database_url else "?"
        database_url = f"{database_url}{separator}sslmode=require"

    return database_url

In [ ]:
url = get_database_url()
print(f"DATABASE_URL:", url)

In [92]:
# ============================================================
# 3. LLM
# ============================================================

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY,
)

In [93]:
# ============================================================
# 4. PYDANTIC MODELS
# ============================================================

class Task(BaseModel):
    id: int
    title: str
    brief: str = Field(..., description="What to cover")


class Plan(BaseModel):
    blog_title: str
    tasks: List[Task]

In [94]:
# ============================================================
# 5. LANGGRAPH STATE
# ============================================================

class State(TypedDict):
    topic: str
    plan: Plan

    # Các worker trả về sections.
    # operator.add sẽ nối các list lại với nhau.
    sections: Annotated[List[str], operator.add]

    final: str

In [95]:
# ============================================================
# 6. ORCHESTRATOR
# ============================================================

def orchestrator(state: State) -> dict:

    plan = llm.with_structured_output(Plan).invoke(
        [
            SystemMessage(
                content=(
                    "Create a blog plan with 5-7 sections "
                    "on the following topic."
                )
            ),
            HumanMessage(
                content=f"Topic: {state['topic']}"
            ),
        ]
    )

    print("PLAN:")
    print(plan.model_dump_json(indent=2))

    return {
        "plan": plan
    }

In [96]:
# ============================================================
# 7. FAN-OUT
# ============================================================

def fanout(state: State):

    return [
        Send(
            "worker",
            {
                "task": task,
                "topic": state["topic"],
                "plan": state["plan"],
            },
        )
        for task in state["plan"].tasks
    ]

In [97]:
# ============================================================
# 8. WORKER
# ============================================================

def worker(payload: dict) -> dict:

    task = payload["task"]
    topic = payload["topic"]
    plan = payload["plan"]

    blog_title = plan.blog_title

    task_title = task.title
    task_brief = task.brief

    section_md = llm.invoke(
        [
            SystemMessage(
                content="Write one clean Markdown section."
            ),
            HumanMessage(
                content=(
                    f"Blog: {blog_title}\n"
                    f"Topic: {topic}\n\n"
                    f"Section: {task_title}\n"
                    f"Brief: {task_brief}\n\n"
                    "Return only the section content in Markdown."
                )
            ),
        ]
    ).content.strip()

    return {
        "sections": [section_md]
    }

In [98]:
# ============================================================
# 9. REDUCER
# ============================================================

def reducer(state: State) -> dict:
    
    title = state["plan"].blog_title
    body = "\n\n".join(state["sections"]).strip()

    final_md = f"# {title}\n\n{body}\n"

    # ---- save to file ----
    filename = title.lower().replace(" ", "_") + ".md"
    output_path = Path(filename)
    output_path.write_text(final_md, encoding="utf-8")

    return {"final": final_md}

In [99]:
# ============================================================
# 10. CHECKPOINTER
# ============================================================

DATABASE_URL = get_database_url()

_conn = psycopg.connect(
    DATABASE_URL,
    autocommit=True,
    row_factory=dict_row
)

checkpointer = PostgresSaver(_conn)
checkpointer.setup()

In [100]:
# ============================================================
# 11. BUILD GRAPH
# ============================================================

graph = StateGraph(State)

graph.add_node("orchestrator", orchestrator)

graph.add_node("worker", worker)

graph.add_node("reducer", reducer)

In [101]:
# ============================================================
# 12. EDGES
# ============================================================

graph.add_edge(START, "orchestrator")

graph.add_conditional_edges(
    "orchestrator",
    fanout,
    ["worker"]
)

graph.add_edge("worker", "reducer")

graph.add_edge("reducer", END)

In [102]:
# ============================================================
# 13. COMPILE
# ============================================================

app = graph.compile(
    checkpointer=checkpointer
)

In [103]:
# ============================================================
# 14. RUN
# ============================================================

config = {
    "configurable": {
        "thread_id": "demo-01"
    }
}

result = app.invoke(
    {
        "topic": "Machine Learning"
    },
    config=config
)

print("\nFINAL:")
print(result["final"])

PLAN:
{
  "blog_title": "Machine Learning",
  "tasks": [
    {
      "id": 1,
      "title": "Introduction to Machine Learning",
      "brief": "Explain what ML is, its purpose, and its impact on modern technology."
    },
    {
      "id": 2,
      "title": "Types of Machine Learning",
      "brief": "Cover supervised, unsupervised, semi-supervised, and reinforcement learning with examples."
    },
    {
      "id": 3,
      "title": "Key Algorithms and Techniques",
      "brief": "Discuss popular algorithms such as linear regression, decision trees, SVMs, neural networks, and clustering."
    },
    {
      "id": 4,
      "title": "Data Preparation and Feature Engineering",
      "brief": "Highlight the importance of data quality, cleaning, scaling, and feature extraction."
    },
    {
      "id": 5,
      "title": "Model Training, Evaluation, and Deployment",
      "brief": "Outline the workflow from training to validation, hyperparameter tuning, and deploying models."
    },
    {